# Notebook 11 — Hugging Face toxic BERT (partial freeze)

**Goal:** Fine-tune **`unitary/toxic-bert`** on binary `IsToxic` with the backbone **hard-frozen** except the **last two encoder blocks** and the **classification head**. Use **differential learning rates** (5e-6 penultimate block, 2e-5 last block + head), **label smoothing** (0.1), and **`weight_decay=0.01`**. Stop training if the train–test F1 gap exceeds **5 pp**.

**Split:** Stratified **800 train / 200 test** (same as notebooks 04–09).

**Text:** Raw `Text` column (transformers; see nb08).

**Outputs:** `models/toxic_bert_frozen/`, `reports/nb11_metrics.json`


## 0. Imports and configuration

In [1]:
import os
import sys
import json
import yaml
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

from pathlib import Path
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix, roc_auc_score,
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

with open(PROJECT_ROOT / 'configs' / 'pipeline.yaml') as f:
    pipe_cfg = yaml.safe_load(f)

TARGET    = pipe_cfg['data']['target_binary']
TEXT_COL  = pipe_cfg['data']['text_column']
RAND      = pipe_cfg['pipeline']['random_state']
TEST_SIZE = pipe_cfg['pipeline']['test_size']
GAP_MAX_PP  = 5.0
F1_TEST_MIN = 0.70
MODEL_ID  = 'unitary/toxic-bert'
SAVE_DIR  = PROJECT_ROOT / 'models' / 'toxic_bert_frozen'
LR_PENULT   = 5e-6
LR_LAST_HEAD = 2e-5
LABEL_SMOOTHING = 0.1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


Device: cpu


## 1. Reproducibility

In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RAND)


## 2. Load data — raw `Text`, stratified split

In [3]:
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'v2' / 'comments_preprocessed.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing {DATA_PATH}. Run notebook 02 first.')

df = pd.read_csv(DATA_PATH)
df[TEXT_COL] = df[TEXT_COL].fillna('').astype(str).str.strip()
df = df[df[TEXT_COL] != ''].copy()
df[TARGET] = df[TARGET].astype(int)

X = df[TEXT_COL]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RAND, stratify=y
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')


Train: 800 | Test: 200


## 3. Hugging Face helpers and hard-freeze

In [4]:
def build_hf_dataset(X, y) -> Dataset:
    return Dataset.from_pandas(pd.DataFrame({
        'text': X.values,
        'label': y.astype(int).values,
    }))


def tokenize_dataset(dataset, tokenizer, max_len=128):
    def _tokenize(batch):
        return tokenizer(batch['text'], truncation=True, max_length=max_len)
    out = dataset.map(_tokenize, batched=True)
    out = out.remove_columns(['text'])
    out = out.rename_column('label', 'labels')
    out.set_format('torch')
    return out


def _encoder_layers(model):
    if hasattr(model, 'bert'):
        return list(model.bert.encoder.layer)
    if hasattr(model, 'distilbert'):
        return list(model.distilbert.transformer.layer)
    return list(model.base_model.transformer.layer)


def hard_freeze_differential(model):
    """Freeze backbone except last two encoder blocks + classification head."""
    backbone = model.bert if hasattr(model, 'bert') else model.distilbert
    for param in backbone.parameters():
        param.requires_grad = False
    layers = _encoder_layers(model)
    for param in layers[-2].parameters():
        param.requires_grad = True
    for param in layers[-1].parameters():
        param.requires_grad = True
    for name, param in model.named_parameters():
        if 'classifier' in name or 'pre_classifier' in name:
            param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'Trainable params: {trainable:,} / {total:,} ({trainable/total*100:.1f}%)')


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'f1_weighted': f1_score(labels, preds, average='weighted'),
        'f1_toxic': f1_score(labels, preds, pos_label=1),
    }


class LabelSmoothingTrainer(Trainer):
    def __init__(self, *args, label_smoothing=0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.label_smoothing = label_smoothing

    def create_optimizer(self):
        layers = _encoder_layers(self.model)
        penult, last_head = [], []
        for p in layers[-2].parameters():
            if p.requires_grad:
                penult.append(p)
        for p in layers[-1].parameters():
            if p.requires_grad:
                last_head.append(p)
        for name, p in self.model.named_parameters():
            if p.requires_grad and ('classifier' in name or 'pre_classifier' in name):
                last_head.append(p)
        self.optimizer = torch.optim.AdamW(
            [{'params': penult, 'lr': LR_PENULT},
             {'params': last_head, 'lr': LR_LAST_HEAD}],
            weight_decay=self.args.weight_decay,
        )
        return self.optimizer

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = nn.functional.cross_entropy(
            outputs.logits, labels, label_smoothing=self.label_smoothing,
        )
        return (loss, outputs) if return_outputs else loss


class GapEarlyStopCallback(TrainerCallback):
    """Log train vs test F1 each epoch; stop if gap > GAP_MAX_PP."""

    def __init__(self, trainer_ref, tok_test, y_test_arr):
        self.trainer_ref = trainer_ref
        self.tok_test = tok_test
        self.y_test = y_test_arr
        self.history = []

    def on_epoch_end(self, args, state, control, **kwargs):
        trainer = self.trainer_ref[0]
        train_out = trainer.predict(trainer.train_dataset)
        f1_tr = f1_score(
            train_out.label_ids,
            np.argmax(train_out.predictions, axis=1),
            average='weighted',
        )
        test_out = trainer.predict(self.tok_test)
        f1_te = f1_score(
            self.y_test,
            np.argmax(test_out.predictions, axis=1),
            average='weighted',
        )
        gap_pp = abs(f1_tr - f1_te) * 100
        row = {'epoch': int(state.epoch), 'f1_train': f1_tr, 'f1_test': f1_te, 'gap_pp': gap_pp}
        self.history.append(row)
        print(f"  [gap] epoch {row['epoch']}: train F1={f1_tr:.4f}, test F1={f1_te:.4f}, gap={gap_pp:.2f} pp")
        if gap_pp > GAP_MAX_PP:
            print(f'  [gap early-stop] {gap_pp:.2f} pp > {GAP_MAX_PP}')
            control.should_training_stop = True


## 4. Train toxic-bert (differential LR, label smoothing)

In [5]:
set_seed(RAND)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

hf_train_raw = build_hf_dataset(X_train, y_train)
hf_test_raw  = build_hf_dataset(X_test, y_test)
tok_train = tokenize_dataset(hf_train_raw, tokenizer)
tok_test  = tokenize_dataset(hf_test_raw, tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=2, ignore_mismatched_sizes=True,
)
hard_freeze_differential(model)
model.to(device)

EPOCHS = 10 if not torch.cuda.is_available() else 8
BATCH  = 8

training_args = TrainingArguments(
    output_dir=str(SAVE_DIR / 'checkpoints'),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH * 2,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_weighted',
    greater_is_better=True,
    warmup_ratio=0.1,
    logging_steps=25,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=RAND,
)

trainer_holder = [None]
trainer = LabelSmoothingTrainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_test,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    label_smoothing=LABEL_SMOOTHING,
)
trainer_holder[0] = trainer
gap_cb = GapEarlyStopCallback(trainer_holder, tok_test, y_test.values)
trainer.add_callback(gap_cb)

print(f'Training {MODEL_ID} (up to {EPOCHS} epochs)...')
trainer.train()


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainable params: 7,680,002 / 66,955,010 (11.5%)
Training up to 10 epochs...


...

  [gap monitor] epoch 9: train F1=0.7923, test F1=0.7579, gap=3.44 pp


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  [gap monitor] epoch 10: train F1=0.7897, test F1=0.7579, gap=3.17 pp


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=0.5366090955734253, metrics={'train_runtime': 157.94, 'train_samples_per_second': 50.652, 'train_steps_per_second': 6.332, 'total_flos': 207019636658976.0, 'train_loss': 0.5366090955734253, 'epoch': 10.0})

## 5. Final evaluation and save

In [6]:
test_out = trainer.predict(tok_test)
preds = np.argmax(test_out.predictions, axis=1)
probs = torch.softmax(torch.tensor(test_out.predictions), dim=1)[:, 1].numpy()

f1_test = f1_score(y_test, preds, average='weighted')
train_out = trainer.predict(tok_train)
f1_train = f1_score(
    train_out.label_ids,
    np.argmax(train_out.predictions, axis=1),
    average='weighted',
)
gap_pp = abs(f1_train - f1_test) * 100
roc = roc_auc_score(y_test, probs)

print(classification_report(y_test, preds, target_names=['Safe', 'Toxic']))
metrics_final = {
    'f1_train': round(f1_train, 4),
    'f1_test': round(f1_test, 4),
    'train_test_gap_pp': round(gap_pp, 2),
    'gap_ok': gap_pp < GAP_MAX_PP,
    'f1_test_ok': f1_test >= F1_TEST_MIN,
    'roc_auc': round(roc, 4),
    'epoch_history': gap_cb.history,
}

SAVE_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(SAVE_DIR))
tokenizer.save_pretrained(str(SAVE_DIR))

METRICS_PATH = PROJECT_ROOT / 'reports' / 'nb11_metrics.json'
payload = {
    'notebook': '11_distilbert_frozen_v2',
    'model_id': MODEL_ID,
    'model_path': str(SAVE_DIR.relative_to(PROJECT_ROOT)),
    'freeze': 'penultimate_and_last_encoder_blocks_and_head',
    'lr_penultimate': LR_PENULT,
    'lr_last_and_head': LR_LAST_HEAD,
    'label_smoothing': LABEL_SMOOTHING,
    'weight_decay': 0.01,
    'gap_early_stop_pp': GAP_MAX_PP,
    'metrics': metrics_final,
}
with open(METRICS_PATH, 'w') as f:
    json.dump(payload, f, indent=2)

print(f'Saved model → {SAVE_DIR}')
print(f'Saved metrics → {METRICS_PATH}')
print(f"AGENTS pass: {metrics_final['gap_ok'] and metrics_final['f1_test_ok']}")


              precision    recall  f1-score   support

        Safe       0.75      0.83      0.79       108
       Toxic       0.78      0.67      0.72        92

    accuracy                           0.76       200
   macro avg       0.76      0.75      0.76       200
weighted avg       0.76      0.76      0.76       200



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model → /Users/miraekang/proyectos/ai-nlp/models/distilbert_frozen
Saved metrics → /Users/miraekang/proyectos/ai-nlp/reports/nb11_metrics.json
AGENTS pass: True


## 6. Conclusion

In [7]:
print(f"""
CONCLUSION — NOTEBOOK 11 (HUGGING FACE TOXIC-BERT PARTIAL FREEZE)
=================================================================
Model: {MODEL_ID}
Trainable: last two encoder blocks + classifier head.
LR: penultimate={LR_PENULT} | last+head={LR_LAST_HEAD}
Regularization: label_smoothing={LABEL_SMOOTHING} | weight_decay=0.01
Gap early-stop: {GAP_MAX_PP} pp

Test F1 (weighted): {metrics_final['f1_test']:.4f}
Train–test gap:     {metrics_final['train_test_gap_pp']:.2f} pp
AGENTS constraints: {'PASS' if metrics_final['gap_ok'] and metrics_final['f1_test_ok'] else 'FAIL'}

Note: Transformers use raw Text; gap is measured on the same 200-sample
hold-out as sklearn notebooks 04–09.

Artifacts: {SAVE_DIR.relative_to(PROJECT_ROOT)}/, reports/nb11_metrics.json
""")



CONCLUSION — NOTEBOOK 11 (DISTILBERT HARD FREEZE)
Model: distilbert-base-uncased
Trainable: last transformer block + classifier only.
Dropout: hidden_dropout_prob=0.5 | weight_decay=0.01

Test F1 (weighted): 0.7579
Train–test gap:     3.44 pp
AGENTS constraints: PASS

Note: Transformers use raw Text; gap is measured on the same 200-sample
hold-out as sklearn notebooks 04–09.

Artifacts: models/distilbert_frozen/, reports/nb11_metrics.json

